In [1]:
import json
import numpy as np
import os

dir_path = 'results_json'
result_files = os.listdir(dir_path)
result_paths = [os.path.join(dir_path, file) for file in result_files]

In [2]:

fold_num = 10
task_num = 4

for path in result_paths:
    metrics: dict[str, list] = {}
    with open(path, 'r', encoding='utf-8') as opened_file:
        data = json.load(opened_file)
        performance = data['performance']
        for fold in range(fold_num):
            R_acc, R_mf1 = [], []
            for task in range(task_num + 1):
                R_acc.append(performance[f'task{task}_fold{fold}']['acc'])
                R_mf1.append(performance[f'task{task}_fold{fold}']['mF1'])
            AP_acc, AP_mF1 = 0, 0
            for task in range(task_num):
                AP_acc += R_acc[task_num][task] / task_num
                AP_mF1 += R_mf1[task_num][task] / task_num
            if 'AP_acc' not in metrics:
                metrics['AP_acc'] = [AP_acc]
                metrics['AP_mf1'] = [AP_mF1]
            else:
                metrics['AP_acc'].append(AP_acc)
                metrics['AP_mf1'].append(AP_mF1)
            BP_acc, BP_mF1 = 0, 0
            for task in range(task_num):
                BP_acc += R_acc[task + 1][task] / task_num
                BP_mF1 += R_mf1[task + 1][task] / task_num
            if 'BP_acc' not in metrics:
                metrics['BP_acc'] = [BP_acc]
                metrics['BP_mf1'] = [BP_mF1]
            else:
                metrics['BP_acc'].append(BP_acc)
                metrics['BP_mf1'].append(BP_mF1)
            BWT_acc, BWT_mF1 = 0, 0
            for task in range(task_num):
                BWT_acc += (R_acc[task_num][task] - R_acc[task + 1][task]) / (task_num - 1)
                BWT_mF1 += (R_mf1[task_num][task] - R_mf1[task + 1][task]) / (task_num - 1)
            if 'BWT_acc' not in metrics:
                metrics['BWT_acc'] = [BWT_acc]
                metrics['BWT_mf1'] = [BWT_mF1]
            else:
                metrics['BWT_acc'].append(BWT_acc)
                metrics['BWT_mf1'].append(BWT_mF1)
            FWT_acc, FWT_mF1 = 0, 0
            for task in range(task_num):
                FWT_acc += (R_acc[task][task] - R_acc[0][task]) / (task_num - 1)
                FWT_mF1 += (R_mf1[task][task] - R_mf1[0][task]) / (task_num - 1)
            if 'FWT_acc' not in metrics:
                metrics['FWT_acc'] = [FWT_acc]
                metrics['FWT_mf1'] = [FWT_mF1]
            else:
                metrics['FWT_acc'].append(FWT_acc)
                metrics['FWT_mf1'].append(FWT_mF1)
            for task in range(task_num):
                if f'First_task_acc_{task}' not in metrics:
                    metrics[f'First_task_acc_{task}'] = []
                    metrics[f'First_task_mf1_{task}'] = []
                    metrics[f'Accum_task_acc_{task}'] = []
                    metrics[f'Accum_task_mf1_{task}'] = []
                metrics[f'First_task_acc_{task}'].append(R_acc[task + 1][0])
                metrics[f'First_task_mf1_{task}'].append(R_mf1[task + 1][0])
                accum_forget_acc, accum_forget_mf1 = 0, 0
                for t in range(task):
                    accum_forget_acc += R_acc[task + 1][t] - R_acc[t + 1][t]
                    accum_forget_mf1 += R_mf1[task + 1][t] - R_mf1[t + 1][t]
                metrics[f'Accum_task_acc_{task}'].append(accum_forget_acc)
                metrics[f'Accum_task_mf1_{task}'].append(accum_forget_mf1)
    metric_statistical = {}
    for (key, values) in metrics.items():
        values = [value * 100 for value in values]
        metric_statistical[key] = {'mean': np.mean(values), 'std': np.std(values)}
    metric_statistical['First_task_acc'] = []
    metric_statistical['First_task_mf1'] = []
    metric_statistical['Accum_task_acc'] = []
    metric_statistical['Accum_task_mf1'] = []
    for i in range(task_num):
        metric_statistical['First_task_acc'].append(metric_statistical[f'First_task_acc_{i}'])
        metric_statistical['First_task_mf1'].append(metric_statistical[f'First_task_mf1_{i}'])
        del metric_statistical[f'First_task_acc_{i}']
        del metric_statistical[f'First_task_mf1_{i}']
        metric_statistical['Accum_task_acc'].append(metric_statistical[f'Accum_task_acc_{i}'])
        metric_statistical['Accum_task_mf1'].append(metric_statistical[f'Accum_task_mf1_{i}'])
        del metric_statistical[f'Accum_task_acc_{i}']
        del metric_statistical[f'Accum_task_mf1_{i}']
    '''
    for (key, values) in metric_statistical.items():
        print(f'{key}: {values}')
    '''
    root, ext = os.path.splitext(path)
    new_path = f'{root}_metrics{ext}'
    with open(new_path, 'w', encoding='utf-8') as opened_file:
        json.dump(metric_statistical, opened_file, indent=4)
    '''
    new_path = f'{root}_table{ext}'
    with open(new_path, 'w', encoding='utf-8') as opened_file:
        def format(mean_std):
            meanv, stdv = mean_std['mean'], mean_std['std']
            opened_file.write(f'& {meanv:.2f}($\pm${stdv:.2f}) ')
        format(metric_statistical['AP_acc'])
        format(metric_statistical['AP_mf1'])
        format(metric_statistical['BWT_acc'])
        format(metric_statistical['BWT_mf1'])
        format(metric_statistical['FWT_acc'])
        format(metric_statistical['FWT_mf1'])
        format(metric_statistical['BP_acc'])
        format(metric_statistical['BP_mf1'])
        opened_file.write('\\\\')
    '''